# YRBS 2007 data cleaning and recoding

This notebook:
1. Loads `../data/raw/YRBS_2007.csv`
2. Replaces common missing-value markers with `NaN`
3. Saves the cleaned data to `../data/processed/yrbs_cleaned.csv`
4. Recodes the requested variables and saves the result to `../data/processed/yrbs_recoded.csv`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


## Data Check

### Selected variables for this project
- **Behavior variable:** `CurrentCigaretteUse`
- **Continuous variable:** `BMIPCT`

### Variable definitions
- `CurrentCigaretteUse` measures current cigarette smoking behavior.
- `BMIPCT` is BMI percentile.

### Valid codes and recoding rule
For `CurrentCigaretteUse`:
- original code **1** = failure
- original codes **2–7** = success

So after recoding:
- **1 = success** = current cigarette use
- **0 = failure** = no current cigarette use

### Missing or invalid values
Missing or invalid values are converted to `NaN` and excluded from analysis.

### Why this matters
This notebook prepares the cleaned and recoded data for EDA and one-sample inference.

In [2]:
RAW_PATH = Path('../data/raw/YRBS_2007.csv')
PROCESSED_DIR = Path('../data/processed')
CLEAN_PATH = PROCESSED_DIR / 'yrbs_cleaned.csv'
RECODED_PATH = PROCESSED_DIR / 'yrbs_recoded.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    'RaceEth', 'HowOldAreYou', 'WhatIsYourSex', 'InWhatGradeAreYou',
    'AreYouHispanicOrLatino', 'WhatIsYourRace',
    'HowTallAreYouWithoutShoesInMeters', 'HowMuchDoYouWeighWithoutShoesInKG',
    'BicyleHelmetUse', 'SeatBeltUse', 'RidingWithADrinkingDriver',
    'DrinkingAndDriving', 'WeaponCarrying', 'GunCarryingPast12Mos',
    'WeaponCarryingAtSchool', 'SafetyConcernsAtSchool',
    'WereThreatenedOrInjuredWithAWeaponOnSchoolProperty',
    'StolenOrDamagedYourProperty', 'PhysicalFighting', 'InjuredIFight',
    'PhysicalFightingAtSchool', 'BoyfriendGirlfriendPhysicallyHurt',
    'ForcedSexualIntercourse', 'SadOrHopeless', 'ConsideredSuicide',
    'MadeASuicidePlan', 'AttemptedSuicide', 'InjuriousSuicide',
    'EverCigaretteUse', 'InitiationSmokingWholeCigarette',
    'CurrentCigaretteUse', 'SmokedMoreThan10Cigarettes',
    'HowObtainedCigarettes', 'SmokeOnSchoolProperty',
    'EverSmokedDailyFor30Days', 'EverSmokingCessation',
    'CurrentSmokelessTobaccoUse',
    'CurrentSmokelessTobaccoOnSchoolProperty', 'CurrentCigarUse',
    'EverAlcoholUse', 'InitiationOfAlcoholUse', 'CurrentAlcoholUse',
    'CurrentBingeDrinking5OrMore', 'SourceOfAlcohol',
    'DrinkAlcoholOnSchoolProperty', 'EverMarijuaUse',
    'InitiationOfMarijuaUse', 'CurrentMarijuaUse',
    'MarijuaOnSchoolProperty', 'EverCocaineUse',
    'CocaineUsePast30Days', 'EverInhalantUse', 'EverHeroinUse',
    'EverMethamphetamineUse', 'EverEcstasyUse', 'EverSteriodUse',
    'IllegalInjectedDrugUse', 'IllegalDrugsAtSchool',
    'EverSexualIntercourse', 'FirstSexualIntercourse',
    'MultipleSexPartners', 'CurrentSexualActivity',
    'AlcoholDrugsAndSex', 'CondomUse', 'BirthControlPillUse',
    'PerceptionOfWeight', 'WeightLoss', 'ExerciseToLoseWeight',
    'ConsumeFewerCaloriesToLoseWeight', 'FastingToLoseWeight',
    'DietPillsToLoseWeight', 'LaxativesToLoseWeight',
    'FruitJuiceDrinking', 'FruitEating', 'GreenSaladEating',
    'PotatoEating', 'CarrotEating', 'OtherVegetableEating',
    'NoSodaDrinking', 'NoMilkDrinking',
    'PhysicalActivity5OrMoreDays', 'TelevisionWatching',
    'ComputerUse', 'PEAttendance', 'SportsTeamParticipation',
    'TaughtAboutHIV', 'Asthma', 'StillHaveAsthma',
    'UsedMotorcycleHelmet', 'EverUsedLSD', 'AerobicExercise',
    'NoerobicExercise', 'MinutesInPEPlayingSports',
    'InjuredWhileExercising', 'HIVTesting', 'SunscreenUse',
    'SunProtection', 'Sleep', 'HealthInGeneral', 'BMIPCT',
    'weight', 'stratum', 'psu'
]


In [3]:
# Read as strings first so coded values are preserved exactly during cleaning.
df = pd.read_csv(RAW_PATH, dtype=str)

# Normalize whitespace in column names and cell values.
df.columns = df.columns.str.strip()
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Replace common missing-value markers with NaN.
missing_markers = ['', 'NA', 'N/A', 'na', 'n/a', 'NULL', 'null', 'None', '.', ' ']
df = df.replace(missing_markers, np.nan)

# Keep only the requested variables that are present in the file, in the requested order.
available_columns = [col for col in EXPECTED_COLUMNS if col in df.columns]
missing_columns = [col for col in EXPECTED_COLUMNS if col not in df.columns]

df = df[available_columns].copy()

# Convert values to numeric where possible.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

df.to_csv(CLEAN_PATH, index=False)

print(f'Cleaned data saved to: {CLEAN_PATH.resolve()}')
print(f'Rows, columns: {df.shape}')
if missing_columns:
    print('Columns from the request that were not found in the raw file:')
    print(missing_columns)
else:
    print('All requested columns were found.')


C:\Users\kubro\AppData\Local\Temp\ipykernel_25336\382532285.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


Cleaned data saved to: C:\Users\kubro\Desktop\project-cycle-2\data\processed\yrbs_cleaned.csv
Rows, columns: (14041, 103)
All requested columns were found.


In [4]:
recoded_df = df.copy()

def recode_binary(series, success_codes, failure_codes):
    series = pd.to_numeric(series, errors='coerce')
    out = pd.Series(np.nan, index=series.index, dtype='float')
    out[series.isin(success_codes)] = 1
    out[series.isin(failure_codes)] = 0
    return out

recoding_rules = {
    'EverCigaretteUse': {'success': [1], 'failure': [2]},
    'SadOrHopeless': {'success': [1], 'failure': [2]},
    'CurrentCigaretteUse': {'success': [2, 3, 4, 5, 6, 7], 'failure': [1]},
    'CurrentAlcoholUse': {'success': [2, 3, 4, 5, 6, 7], 'failure': [1]},
}

for variable, rule in recoding_rules.items():
    if variable in recoded_df.columns:
        recoded_df[variable] = recode_binary(
            recoded_df[variable],
            success_codes=rule['success'],
            failure_codes=rule['failure']
        )

recoded_df.to_csv(RECODED_PATH, index=False)
print(f'Recoded data saved to: {RECODED_PATH.resolve()}')
recoded_df[['EverCigaretteUse', 'SadOrHopeless', 'CurrentCigaretteUse', 'CurrentAlcoholUse']].head()


Recoded data saved to: C:\Users\kubro\Desktop\project-cycle-2\data\processed\yrbs_recoded.csv


,EverCigaretteUse,SadOrHopeless,CurrentCigaretteUse,CurrentAlcoholUse
0,1.0,1.0,1.0,NaN
1,1.0,0.0,NaN,NaN
2,NaN,1.0,NaN,NaN
3,1.0,1.0,0.0,0.0
4,NaN,1.0,0.0,NaN


In [5]:
print("Final sample size for behavior analysis:", df["CurrentCigaretteUse"].notna().sum())
print("Final sample size for continuous analysis:", df["BMIPCT"].notna().sum())

Final sample size for behavior analysis: 13323
Final sample size for continuous analysis: 13062
